In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from multimodal_fin.financial.pipeline import EventPipeline
from multimodal_fin.financial.price_preprocessor import PricePreprocessor

In [2]:
price_file = "../data/financial/price_2019_2025.csv"
companies_folder = "../data/financial/companies_closes"
conference_file = "../data/financial/conference_data.csv"

preprocessor = PricePreprocessor(price_file, companies_folder)

pipeline = EventPipeline(
    conference_file=conference_file,
    prices_folder=companies_folder,
    market_ticker="SPGI" # S&P500 index as market model, it could be NASDAQ e.g if data is available
)

In [3]:
preprocessor.split_by_ticker()
events = pipeline.run() 

2026-05-26 21:57:35 - multimodal_fin.financial.price_preprocessor - INFO - Loading master price file: ../data/financial/price_2019_2025.csv
2026-05-26 21:57:35 - multimodal_fin.financial.price_preprocessor - INFO - Found 217 unique tickers.
2026-05-26 21:57:39 - multimodal_fin.financial.price_preprocessor - INFO - Finished splitting dataset into 217 CSV files.
2026-05-26 21:57:39 - multimodal_fin.financial.calculator - ERROR - [AAPL] No stock data in estimation window for event: Q3 2018 
2026-05-26 21:57:39 - multimodal_fin.financial.calculator - ERROR - [AAPL] No stock data in estimation window for event: Q4 2018 
2026-05-26 21:57:39 - multimodal_fin.financial.calculator - ERROR - [AAPL] No stock data in estimation window for event: Q1 2019 
2026-05-26 21:57:39 - multimodal_fin.financial.company_events - WARNING - [ABT] Skipping 2021-02-09: -17 days available (< 30)
2026-05-26 21:57:39 - multimodal_fin.financial.calculator - ERROR - [ACN] No stock data in estimation window for event: 

In [6]:
def format_window_name(t1, t2):
    """
    Convierte una ventana como (-7, 7) en un sufijo limpio: m7_p7
    """
    def fmt(x):
        return f"m{abs(x)}" if x < 0 else f"p{x}"
    return f"{fmt(t1)}_{fmt(t2)}"


def extract_financial_outcomes_from_event(event, windows=[(-1, 1), (-3, 3), (-7, 7)]):
    """
    Extrae outcomes financieros de un objeto Event.
    
    Devuelve una fila con:
    - ticker, year, quarter, event_date
    - alpha, beta, n_estimation_obs
    - CAR por ventana
    - absolute CAR por ventana
    - volatility of abnormal returns por ventana
    """

    row = {
        "ticker": event.ticker,
        "company_name": event.company_name,
        "year": int(event.year) if event.year is not None else np.nan,
        "quarter": event.quarter,
        "event_date": event.event_date,
        "estimation_start": event.estimation_start,
        "estimation_end": event.estimation_end,
        "gap": event.gap,
    }

    if event.result is None or event.result.df_results is None:
        row["has_event_result"] = False
        row["alpha"] = np.nan
        row["beta"] = np.nan
        row["n_estimation_obs"] = np.nan

        for t1, t2 in windows:
            suffix = format_window_name(t1, t2)
            row[f"CAR_{suffix}"] = np.nan
            row[f"abs_CAR_{suffix}"] = np.nan
            row[f"AR_vol_{suffix}"] = np.nan
            row[f"mean_AR_{suffix}"] = np.nan

        return row

    row["has_event_result"] = True
    row["alpha"] = event.result.alpha
    row["beta"] = event.result.beta
    row["n_estimation_obs"] = event.result.n_estimation_obs

    df_res = event.result.df_results.copy()

    for t1, t2 in windows:
        suffix = format_window_name(t1, t2)

        dfw = df_res[(df_res["t"] >= t1) & (df_res["t"] <= t2)].copy()

        if dfw.empty:
            car = np.nan
            abs_car = np.nan
            ar_vol = np.nan
            mean_ar = np.nan
        else:
            car = float(dfw["AR"].sum())
            abs_car = abs(car)
            ar_vol = float(dfw["AR"].std())
            mean_ar = float(dfw["AR"].mean())

        row[f"CAR_{suffix}"] = car
        row[f"abs_CAR_{suffix}"] = abs_car
        row[f"AR_vol_{suffix}"] = ar_vol
        row[f"mean_AR_{suffix}"] = mean_ar

    return row

In [ ]:
data = pd.read_csv("../data/full_paths.csv")
companies = data["company"].unique().tolist()

event_windows = [(-7, 7), (-7, 30), (-7, 60), (-7, 90), (0, 3), (0, 7)]
financial_rows = []

for ticker in companies:
    try:
        company_events_df = pipeline.list_company_events(ticker, return_df=True)

        for _, ev_row in company_events_df.iterrows():
            year = int(ev_row["year"])
            quarter = ev_row["quarter"]

            try:
                event = pipeline.get_event(
                    ticker,
                    year=year,
                    quarter=quarter,
                    events=events
                )

                if event is None:
                    print(f"Missing event object: {ticker} {year} {quarter}")
                    continue

                financial_rows.append(
                    extract_financial_outcomes_from_event(
                        event,
                        windows=event_windows
                    )
                )

            except Exception as e:
                print(f"Skipping event {ticker} {year} {quarter}: {e}")

    except Exception as e:
        print(f"Skipping ticker {ticker}: {e}")

event_financial_df = pd.DataFrame(financial_rows)
event_financial_df.to_csv('event_financial_df.csv')

display(event_financial_df.head())
print(event_financial_df.shape)

Skipping event SYK 2024 Q2: No event found for SYK 2024 Q2.
Skipping event PPG 2022 Q4: No event found for PPG 2022 Q4.
Skipping event JPM 2020 Q2: No event found for JPM 2020 Q2.
Skipping event BKNG 2019 Q2: No event found for BKNG 2019 Q2.
Skipping event NVDA 2020 Q3: No event found for NVDA 2020 Q3.
Skipping event ABT 2021 Q1: No event found for ABT 2021 Q1.
Skipping event GOOG 2019 Q1: No event found for GOOG 2019 Q1.
Skipping event FOXA 2021 Q2: No event found for FOXA 2021 Q2.
Skipping event AVB 2021 Q1: No event found for AVB 2021 Q1.
Skipping event AVGO 2021 Q3: No event found for AVGO 2021 Q3.
Skipping event AVGO 2020 Q1: No event found for AVGO 2020 Q1.
Skipping event LMT 2024 Q4: No event found for LMT 2024 Q4.
Skipping event PARA 2019 Q1: No event found for PARA 2019 Q1.
Skipping event GD 2020 Q2: No event found for GD 2020 Q2.
Skipping event CL 2019 Q3: No event found for CL 2019 Q3.


,ticker,company_name,year,quarter,event_date,estimation_start,estimation_end,gap,has_event_result,alpha,beta,n_estimation_obs,CAR_p0_p3,abs_CAR_p0_p3,AR_vol_p0_p3,mean_AR_p0_p3,CAR_p0_p7,abs_CAR_p0_p7,AR_vol_p0_p7,mean_AR_p0_p7
0,KMI,"Kinder Morgan, Inc.",2020,Q3,2020-10-21,2020-03-18,2020-09-14,30,True,-0.002981,0.996489,125.0,0.024800,0.024800,0.026730,0.008267,-0.011296,0.011296,0.020549,-0.001883
1,KMI,"Kinder Morgan, Inc.",2020,Q4,2021-01-20,2020-10-14,2020-12-14,30,True,0.003304,-0.073431,43.0,-0.024347,0.024347,0.015214,-0.008116,-0.069531,0.069531,0.014258,-0.011589
2,KMI,"Kinder Morgan, Inc.",2021,Q1,2021-04-21,2021-01-13,2021-03-15,30,True,0.001881,0.092759,42.0,0.005964,0.005964,0.022370,0.001988,0.040847,0.040847,0.016739,0.006808
3,KMI,"Kinder Morgan, Inc.",2021,Q2,2021-07-21,2021-04-14,2021-06-14,30,True,0.003498,-0.024917,43.0,-0.016107,0.016107,0.021753,-0.005369,-0.004248,0.004248,0.017884,-0.000708
4,KMI,"Kinder Morgan, Inc.",2021,Q3,2021-10-20,2021-07-14,2021-09-13,30,True,-0.002956,0.185112,43.0,-0.034082,0.034082,0.043252,-0.011361,-0.051287,0.051287,0.029661,-0.008548


(3674, 20)
